# 02 - EDA e Storytelling Executivo (Datathon Passos Mágicos)

Este notebook transforma a análise exploratória em narrativa gerencial com foco em decisão.
A estrutura segue o enunciado oficial e responde as 11 perguntas de negócio com base em evidências da base Silver/Gold.


## 1) Contexto do problema

A Passos Mágicos acompanha alunos por indicadores acadêmicos, de engajamento, psicossociais e psicopedagógicos.
Quando esses dados ficam dispersos, a tomada de decisão tende a ser reativa.

**Pergunta central do projeto:**

> Quais alunos ou grupos apresentam maior sinal de risco educacional e quais fatores ajudam a explicar esse risco?


## 2) Objetivo da análise

Esta EDA busca:

- identificar padrões de desempenho e risco ao longo do tempo;
- comparar grupos (ano, fase, pedra e combinações de indicadores);
- destacar sinais que ajudam a priorizar acompanhamento preventivo.

Essa leitura prepara o caminho para a modelagem preditiva no notebook `03_modelagem_risco.ipynb`.


In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from IPython.display import display, Markdown
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="viridis")
plt.rcParams["figure.figsize"] = (10, 5)

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

silver_path = PROJECT_ROOT / "data" / "silver" / "base_unificada_validada_enriquecida.csv"
gold_path = PROJECT_ROOT / "data" / "gold" / "base_modelagem_risco.csv"

silver = pd.read_csv(silver_path, low_memory=False)
gold = pd.read_csv(gold_path, low_memory=False)

# Foco oficial do case: anos 2022-2024
silver_eda = silver[silver["ano_referencia"].between(2022, 2024)].copy()
gold_eda = gold[gold["ano_referencia"].between(2022, 2024)].copy()

story_facts = {}

print("Silver EDA:", silver_eda.shape)
print("Gold EDA:", gold_eda.shape)


## 3) Preparação e confiabilidade da base

- A consolidação foi realizada por `ra + ano_referencia`.
- Houve reconciliação entre base antiga e base nova para manter consistência histórica.
- A camada Gold concentra os campos de análise/modelagem e o `target_risco` temporal.

Nesta etapa, o foco é **analítico**: usar a base já tratada para gerar evidência de negócio.


In [ ]:
key_silver_dup = silver_eda.duplicated(subset=["ra", "ano_referencia"]).sum()
key_gold_dup = gold_eda.duplicated(subset=["ra", "ano_referencia"]).sum()

quality = pd.DataFrame(
    {
        "base": ["silver_eda", "gold_eda"],
        "linhas": [len(silver_eda), len(gold_eda)],
        "alunos_unicos": [silver_eda["ra"].nunique(), gold_eda["ra"].nunique()],
        "chaves_duplicadas_ra_ano": [int(key_silver_dup), int(key_gold_dup)],
        "anos": [
            ", ".join(map(str, sorted(silver_eda["ano_referencia"].dropna().unique()))),
            ", ".join(map(str, sorted(gold_eda["ano_referencia"].dropna().unique()))),
        ],
    }
)

display(quality)

story_facts["taxa_risco_media"] = float(gold_eda["target_risco"].mean()) * 100
story_facts["n_alunos"] = int(gold_eda["ra"].nunique())


## 4) Visão geral dos dados

Leitura executiva esperada:

- cobertura temporal dos anos analisados;
- distribuição por fase/pedra para entender concentração;
- volume de alunos para sustentar inferências gerenciais.


In [ ]:
overview = {
    "registros_silver": len(silver_eda),
    "registros_gold": len(gold_eda),
    "alunos_unicos_silver": silver_eda["ra"].nunique(),
    "alunos_unicos_gold": gold_eda["ra"].nunique(),
    "anos_silver": sorted(silver_eda["ano_referencia"].dropna().unique().tolist()),
    "anos_gold": sorted(gold_eda["ano_referencia"].dropna().unique().tolist()),
    "fases_distintas": silver_eda["fase"].dropna().nunique(),
    "turmas_distintas": silver_eda["turma"].dropna().nunique(),
    "pedras_distintas": silver_eda["pedra"].dropna().nunique(),
}

display(pd.DataFrame([overview]))

# Coverage by year from Silver (2022-2024 available).
cov_alunos_ano = silver_eda.groupby("ano_referencia", as_index=False).agg(
    registros=("ra", "size"),
    alunos=("ra", "nunique"),
)

# Risk by year from Gold (temporal target).
risco_ano = gold_eda.groupby("ano_referencia", as_index=False).agg(
    registros_gold=("ra", "size"),
    risco_medio=("target_risco", "mean"),
)
risco_ano["risco_medio"] = risco_ano["risco_medio"] * 100

resumo_ano = cov_alunos_ano.merge(risco_ano, on="ano_referencia", how="left")
display(resumo_ano.round(2))

display(Markdown(
    "**Nota importante:** o ano de 2024 aparece na cobertura (Silver), mas pode n\u00e3o aparecer "
    "na taxa de risco (Gold) porque o `target_risco` depende da compara\u00e7\u00e3o com o ano seguinte (`t+1`)."
))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.barplot(data=cov_alunos_ano, x="ano_referencia", y="alunos", ax=axes[0], color="#4c78a8")
axes[0].set_title("Cobertura de alunos por ano (Silver)")
axes[0].set_ylabel("Alunos \u00fanicos")
axes[0].set_xlabel("Ano")

sns.lineplot(data=risco_ano, x="ano_referencia", y="risco_medio", marker="o", ax=axes[1], color="#e45756")
axes[1].set_title("Taxa m\u00e9dia de risco por ano (Gold)")
axes[1].set_ylabel("Risco m\u00e9dio (%)")
axes[1].set_xlabel("Ano")
plt.tight_layout()
plt.show()


## 5) Respostas às 11 perguntas do enunciado

As respostas abaixo seguem uma narrativa executiva: problema de negócio, evidência nos dados, leitura dos resultados e implicação prática para a Passos Mágicos.

Cada pergunta foi tratada como uma decisão de gestão: o objetivo não é apenas mostrar gráficos, mas transformar os dados em sinais para acompanhamento pedagógico, psicossocial e preventivo.

In [ ]:
def add_ian_band(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    # Faixa proxy para leitura executiva do IAN
    out["faixa_ian"] = pd.cut(
        out["ian"],
        bins=[-np.inf, 4, 6, np.inf],
        labels=["Severa", "Moderada", "Adequada"],
    )
    return out


def safe_qcut(series: pd.Series, label: str, q: int = 4) -> pd.Series:
    s = pd.to_numeric(series, errors="coerce")
    valid = s.dropna()
    if valid.nunique() <= 1:
        out = pd.Series(index=s.index, dtype="object")
        out.loc[valid.index] = f"{label}_Q1"
        return out

    q_eff = min(q, int(valid.nunique()))
    ranked = valid.rank(method="first")
    labels = [f"{label}_Q{i}" for i in range(1, q_eff + 1)]
    buckets = pd.qcut(ranked, q=q_eff, labels=labels)

    out = pd.Series(index=s.index, dtype="object")
    out.loc[valid.index] = buckets.astype(str)
    return out


## Pergunta 1 — Perfil de adequação de nível (IAN)

**Pergunta de negócio:**  
Qual é o perfil geral de defasagem/adequação de nível dos alunos e como ele se distribui ao longo dos anos?

**Evidência analisada:**  
Colunas `ian` e `ano_referencia`, organizadas em faixas analíticas de adequação (`Severa`, `Moderada`, `Adequada`).

**Tabela/gráfico:**  
Tabela de alunos por faixa de IAN e gráfico de distribuição percentual por ano.

**Leitura dos dados:**  
A leitura executiva é feita a partir da faixa predominante por ano e da mudança de composição entre grupos.

**Implicação prática:**  
A Passos Mágicos pode usar essa visão para localizar grupos com maior concentração de defasagem e antecipar intervenções.

In [ ]:
q1 = add_ian_band(silver_eda)

tab_q1 = (
    q1.groupby(["ano_referencia", "faixa_ian"]).size().reset_index(name="alunos")
    .pivot(index="ano_referencia", columns="faixa_ian", values="alunos")
    .fillna(0)
    .astype(int)
)
display(tab_q1)

tab_q1_pct = tab_q1.div(tab_q1.sum(axis=1), axis=0) * 100
ax = tab_q1_pct.plot(kind="bar", stacked=True)
ax.set_ylabel("% de alunos")
ax.set_title("Distribuição de faixas de IAN por ano")
plt.show()

faixa_pred = tab_q1_pct.idxmax(axis=1).to_dict()
story_facts["q1_faixa_pred"] = faixa_pred

display(Markdown(
    f"**Leitura dos dados / interpretação executiva:** a faixa predominante por ano foi {faixa_pred}. "
    "Isso mostra como o perfil de adequação se distribui ao longo do ciclo."
))

display(Markdown(
    "**Implicação prática:** monitorar periodicamente a composição dessas faixas por turma/fase "
    "para antecipar intervenções em grupos com maior concentração de defasagem."
))


## Pergunta 2 — Desempenho acadêmico por fase analítica (IDA)

**Pergunta de negócio:**  
O IDA médio está melhorando, estagnado ou caindo quando a comparação por fase é feita de forma metodologicamente confiável?

**Evidência analisada:**  
Colunas `ida`, `ano_referencia` e a dimensão padronizada `fase_analitica`, criada a partir do diagnóstico de qualidade da coluna `fase`.

**Tabela/gráfico:**  
Diagnóstico de qualidade da variável `fase`, tabela de IDA médio por `fase_analitica` e gráfico temporal do IDA.

**Leitura dos dados:**  
A leitura considera somente categorias padronizadas, preservando `NaN` quando não há observação válida, IDA disponível ou categoria comparável naquele ano.

**Implicação prática:**  
A correção evita comparar turma como se fosse fase e melhora a confiabilidade da tomada de decisão por ciclo/série.

## Diagnóstico de qualidade da variável fase

O primeiro gráfico de IDA por fase parecia indicar uma grande ausência de dados entre 2022, 2023 e 2024. O diagnóstico mostrou que a principal causa não era apenas perda de informação, mas uma inconsistência de nomenclatura da variável `fase`.

A coluna original mudou de formato ao longo dos anos: em 2022, aparecia como valores numéricos (`1`, `1.0`, `2`, `2.0`); em 2023, como categorias textuais (`FASE 1`, `FASE 2`, `ALFA`); e em 2024, parte dos valores representava turma, como `1A`, `1B`, `4M`, `7E` e `8E`.

Por isso, comparar diretamente a coluna `fase` original seria tecnicamente frágil. A solução foi criar a dimensão `fase_analitica`, separando a leitura de fase/série/ciclo da informação de turma (`turma_extraida`). Essa etapa reduz o risco de interpretações incorretas e melhora a confiabilidade da comparação temporal do IDA.

O diagnóstico mostrou que a alta quantidade de valores ausentes na matriz fase x ano não representava apenas falta de dados. Parte importante do problema vinha da mudança de nomenclatura da variável `fase` entre os anos e da mistura entre fase e turma, principalmente em 2024. Por isso, foi criada uma dimensão analítica padronizada, separando a leitura de fase da informação de turma. Essa etapa reduz o risco de interpretações incorretas e melhora a confiabilidade da comparação temporal do IDA.

Os `NaN` restantes não foram preenchidos artificialmente. Eles indicam ausência real de observação válida, ausência de IDA ou inexistência daquela categoria analítica em determinado ano.


In [ ]:
import re
import unicodedata

COLUNA_FASE_Q2 = "fase"
COLUNA_IDA_Q2 = "ida"

def normalizar_texto(valor):
    if pd.isna(valor):
        return np.nan

    texto = str(valor).strip().upper()
    texto = unicodedata.normalize("NFKD", texto)
    texto = "".join(c for c in texto if not unicodedata.combining(c))
    texto = re.sub(r"\s+", " ", texto)
    texto = re.sub(r"\.0$", "", texto)
    return texto


def decompor_fase(valor):
    if pd.isna(valor):
        return pd.Series(
            {
                "fase_norm": np.nan,
                "fase_analitica": np.nan,
                "turma_extraida": np.nan,
                "tipo_categoria_fase": "NULO",
            }
        )

    v = normalizar_texto(valor)
    v_sem_fase = re.sub(r"^FASE\s*", "", v).strip()

    if "ALFA" in v_sem_fase:
        return pd.Series(
            {
                "fase_norm": v,
                "fase_analitica": "ALFA",
                "turma_extraida": np.nan,
                "tipo_categoria_fase": "AGRUPAMENTO",
            }
        )

    if re.fullmatch(r"\d+", v_sem_fase):
        return pd.Series(
            {
                "fase_norm": v,
                "fase_analitica": f"FASE {int(v_sem_fase)}",
                "turma_extraida": np.nan,
                "tipo_categoria_fase": "FASE_SERIE_CICLO",
            }
        )

    m = re.fullmatch(r"(\d+)\s*([A-Z])", v_sem_fase)
    if m:
        return pd.Series(
            {
                "fase_norm": v,
                "fase_analitica": f"FASE {int(m.group(1))}",
                "turma_extraida": f"{int(m.group(1))}{m.group(2)}",
                "tipo_categoria_fase": "TURMA_MISTURADA_NA_FASE",
            }
        )

    return pd.Series(
        {
            "fase_norm": v,
            "fase_analitica": f"OUTROS/NAO_CLASSIFICADO - {v}",
            "turma_extraida": np.nan,
            "tipo_categoria_fase": "OUTROS/NAO_CLASSIFICADO",
        }
    )


fase_quality = silver_eda[["ra", "ano_referencia", COLUNA_FASE_Q2, "turma", COLUNA_IDA_Q2]].copy()
fase_quality = fase_quality.rename(columns={COLUNA_FASE_Q2: "fase_original", COLUNA_IDA_Q2: "ida"})
fase_quality = pd.concat([fase_quality, fase_quality["fase_original"].apply(decompor_fase)], axis=1)

categorias_fase_original = fase_quality["fase_original"].nunique(dropna=True)
categorias_fase_analitica = fase_quality["fase_analitica"].nunique(dropna=True)

resumo_metodologico = pd.DataFrame(
    [
        {
            "Item avaliado": "Categorias únicas de fase",
            "Antes": categorias_fase_original,
            "Depois": categorias_fase_analitica,
            "Impacto analítico": "Redução da fragmentação",
        },
        {
            "Item avaliado": "Coluna original",
            "Antes": COLUNA_FASE_Q2,
            "Depois": "fase_analitica",
            "Impacto analítico": "Comparação temporal mais confiável",
        },
        {
            "Item avaliado": "Informação de turma",
            "Antes": "Misturada em fase",
            "Depois": "Separada em turma_extraida",
            "Impacto analítico": "Evita comparar turma como fase",
        },
        {
            "Item avaliado": "NaN no gráfico",
            "Antes": "Alto e parcialmente artificial",
            "Depois": "Mantido apenas quando aplicável",
            "Impacto analítico": "Evita preenchimento indevido",
        },
    ]
)

display(Markdown(f"**Coluna usada originalmente no gráfico:** `{COLUNA_FASE_Q2}`"))
display(Markdown("**Tabela-resumo da correção metodológica**"))
display(resumo_metodologico)

diagnostico_resumo = (
    fase_quality.groupby("ano_referencia")
    .agg(
        registros=("ra", "count"),
        fases_originais=("fase_original", "nunique"),
        fases_padronizadas=("fase_analitica", "nunique"),
        pct_fase_nula=("fase_original", lambda s: s.isna().mean() * 100),
        pct_ida_nulo=("ida", lambda s: s.isna().mean() * 100),
    )
    .round(2)
)

display(Markdown("**Resumo de qualidade por ano**"))
display(diagnostico_resumo)

top_30_fase_original = (
    fase_quality["fase_original"]
    .value_counts(dropna=False)
    .rename_axis("fase_original")
    .reset_index(name="registros")
    .head(30)
)
display(Markdown("**Top 30 valores mais frequentes da fase original**"))
display(top_30_fase_original)

valores_unicos_por_ano = (
    fase_quality.groupby("ano_referencia")["fase_original"]
    .agg(lambda s: sorted([str(v) for v in s.dropna().unique()]))
    .reset_index(name="valores_unicos_fase_original")
)
display(Markdown("**Valores únicos da fase original por ano**"))
display(valores_unicos_por_ano)

matriz_presenca_original = pd.crosstab(
    fase_quality["fase_original"].fillna("NULO"),
    fase_quality["ano_referencia"],
)
matriz_presenca_original = matriz_presenca_original.loc[
    matriz_presenca_original.sum(axis=1).sort_values(ascending=False).index
]
display(Markdown("**Matriz de presença da fase original por ano**"))
display(matriz_presenca_original)

mapeamento_semantico = (
    fase_quality.groupby(
        ["fase_analitica", "fase_norm", "tipo_categoria_fase", "turma_extraida"],
        dropna=False,
    )
    .agg(
        registros=("ra", "count"),
        anos=("ano_referencia", lambda s: ", ".join(map(str, sorted(s.dropna().unique())))),
    )
    .reset_index()
    .sort_values(["fase_analitica", "registros"], ascending=[True, False])
)
display(Markdown("**Mapeamento da fase original para fase analítica padronizada**"))
display(mapeamento_semantico)

matriz_presenca_padronizada = pd.crosstab(
    fase_quality["fase_analitica"].fillna("NULO"),
    fase_quality["ano_referencia"],
)
display(Markdown("**Matriz de presença da fase analítica por ano**"))
display(matriz_presenca_padronizada)

tipos_categoria = pd.crosstab(
    fase_quality["tipo_categoria_fase"],
    fase_quality["ano_referencia"],
)
display(Markdown("**Separação entre fase/série/ciclo, turma misturada, agrupamento e nulos**"))
display(tipos_categoria)


### IDA médio por fase analítica padronizada

A partir do diagnóstico, a comparação temporal passa a usar `fase_analitica`, e não mais a coluna `fase` original como dimensão principal. Essa escolha reduz a fragmentação entre valores como `1`, `1.0`, `FASE 1` e `1A`, mantendo `turma_extraida` separada quando a coluna original trazia uma turma misturada com fase.


In [ ]:
ida_fase_ano = (
    fase_quality.dropna(subset=["ida", "fase_analitica"])
    .groupby(["fase_analitica", "ano_referencia"], as_index=False)
    .agg(
        ida_medio=("ida", "mean"),
        qtd_registros=("ida", "size"),
    )
)

pivot_ida = ida_fase_ano.pivot(
    index="fase_analitica",
    columns="ano_referencia",
    values="ida_medio",
)

pivot_volume = ida_fase_ano.pivot(
    index="fase_analitica",
    columns="ano_referencia",
    values="qtd_registros",
)

display(Markdown("**IDA médio por fase analítica e ano**"))
display(pivot_ida.round(2))

display(Markdown("**Volume de registros com IDA por fase analítica e ano**"))
display(pivot_volume.fillna(0).astype(int))

volume_total = (
    ida_fase_ano.groupby("fase_analitica")["qtd_registros"]
    .sum()
    .sort_values(ascending=False)
)
criterio_min_volume = 10
fases_validas = volume_total[volume_total >= criterio_min_volume].index
plot_df = ida_fase_ano[ida_fase_ano["fase_analitica"].isin(fases_validas)].copy()
plot_df = plot_df.sort_values(["fase_analitica", "ano_referencia"])

display(Markdown(
    f"**Critério do gráfico:** exibir fases analíticas com pelo menos {criterio_min_volume} registros com IDA no período. "
    f"Categorias exibidas: {len(fases_validas)} de {ida_fase_ano['fase_analitica'].nunique()}."
))

plt.figure(figsize=(12, 5))
sns.lineplot(
    data=plot_df,
    x="ano_referencia",
    y="ida_medio",
    hue="fase_analitica",
    marker="o",
)

ida_ano = (
    fase_quality.dropna(subset=["ida"])
    .groupby("ano_referencia", as_index=False)["ida"]
    .mean()
    .rename(columns={"ida": "ida_medio"})
)
sns.lineplot(
    data=ida_ano,
    x="ano_referencia",
    y="ida_medio",
    color="black",
    linewidth=2.5,
    marker="o",
    label="Média geral",
)

plt.title("IDA médio por fase analítica e ano")
plt.ylabel("IDA médio")
plt.xlabel("Ano")
plt.xticks(sorted(plot_df["ano_referencia"].dropna().unique()))
plt.legend(title="Fase analítica", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

presenca_com_ida = pivot_volume.fillna(0).gt(0).sum(axis=1)
fases_comparacao_temporal = sorted(presenca_com_ida[presenca_com_ida >= 2].index.tolist())
fases_cautela = sorted(presenca_com_ida[presenca_com_ida < 2].index.tolist())
ida_delta = float(ida_ano["ida_medio"].iloc[-1] - ida_ano["ida_medio"].iloc[0])
story_facts["q2_ida_delta"] = ida_delta
story_facts["q2_fases_comparacao_temporal"] = fases_comparacao_temporal
story_facts["q2_fases_cautela"] = fases_cautela


### Leitura executiva do gráfico corrigido

O gráfico mostra o IDA médio por ano usando a dimensão `fase_analitica`, com a linha preta representando a média geral da base. A leitura agora está baseada em fase/série/ciclo padronizado, não na coluna `fase` original.

As fases com comparação temporal mais confiável são aquelas com IDA válido em pelo menos dois anos: `ALFA`, `FASE 1`, `FASE 2`, `FASE 3`, `FASE 4`, `FASE 5`, `FASE 6` e `FASE 7`. Entre elas, `FASE 1` a `FASE 7` sustentam leitura mais ampla por aparecerem nos três anos analisados; `ALFA` deve ser lida a partir de 2023.

Categorias como `FASE 0`, `FASE 8` e `FASE 9` exigem cautela. `FASE 0` aparece concentrada em 2022; `FASE 8` tem pouca observação válida de IDA no recorte; e `FASE 9` aparece na padronização, mas não sustenta leitura de IDA médio por falta de IDA válido no período analisado.

A correção é metodológica, não visual: os vazios restantes continuam vazios porque representam ausência real de observação comparável, ausência de IDA ou inexistência daquela fase analítica em determinado ano. A padronização apenas evita que turma seja comparada como se fosse fase.


## Pergunta 3 — Engajamento e desempenho (IEG, IDA e IPV)

**Pergunta de negócio:**  
O engajamento do aluno se relaciona com desempenho acadêmico e ponto de virada?

**Evidência analisada:**  
Colunas `ieg`, `ida` e `ipv`, com correlações e dispersões entre indicadores.

**Tabela/gráfico:**  
Matriz de correlação e gráficos de relação entre IEG, IDA e IPV.

**Leitura dos dados:**  
A leitura usa a intensidade das correlações calculadas, sem assumir causalidade direta.

**Implicação prática:**  
Quando engajamento e desempenho caminham juntos, o IEG pode funcionar como sinal de atenção antecipada para acompanhamento pedagógico.

In [ ]:
q3 = silver_eda[["ieg", "ida", "ipv"]].dropna()

corr_q3 = q3.corr(numeric_only=True)
display(corr_q3.round(3))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.regplot(data=q3, x="ieg", y="ida", scatter_kws={"alpha": 0.3}, ax=axes[0])
axes[0].set_title("IEG vs IDA")
sns.regplot(data=q3, x="ieg", y="ipv", scatter_kws={"alpha": 0.3}, ax=axes[1])
axes[1].set_title("IEG vs IPV")
plt.tight_layout()
plt.show()

c_ida = float(corr_q3.loc["ieg", "ida"])
c_ipv = float(corr_q3.loc["ieg", "ipv"])
story_facts["q3_corr"] = {"ieg_ida": c_ida, "ieg_ipv": c_ipv}

display(Markdown(
    f"**Leitura dos dados / interpretação executiva:** a correlação IEG-IDA foi {c_ida:.3f} e IEG-IPV foi {c_ipv:.3f}. "
    "Esses sinais ajudam a avaliar quanto o engajamento acompanha resultados educacionais."
))

display(Markdown(
    "**Implicação prática:** incluir indicadores de engajamento no painel de risco, "
    "não apenas notas acadêmicas."
))


## Pergunta 4 — Autoavaliação e alinhamento com desempenho (IAA)

**Pergunta de negócio:**  
A percepção do aluno sobre si mesmo está coerente com seu desempenho e engajamento?

**Evidência analisada:**  
Quartis de `iaa` comparados com médias de `ida` e `ieg`.

**Tabela/gráfico:**  
Tabela por faixa de IAA e gráfico comparando IDA e IEG por quartil de autoavaliação.

**Leitura dos dados:**  
A comparação mostra se maiores níveis de autoavaliação acompanham melhores sinais acadêmicos e de engajamento.

**Implicação prática:**  
Desalinhamentos entre autoavaliação, desempenho e engajamento podem orientar conversas pedagógicas e apoio individual.

In [ ]:
q4 = silver_eda[["iaa", "ida", "ieg"]].dropna().copy()
q4["faixa_iaa"] = safe_qcut(q4["iaa"], label="IAA", q=4)

tab_q4 = q4.groupby("faixa_iaa", as_index=False).agg(
    iaa_media=("iaa", "mean"),
    ida_media=("ida", "mean"),
    ieg_media=("ieg", "mean"),
    n=("iaa", "size"),
)

display(tab_q4.round(3))

plt.figure(figsize=(10, 4))
plot_df = tab_q4.melt(
    id_vars=["faixa_iaa", "n"],
    value_vars=["ida_media", "ieg_media"],
    var_name="indicador",
    value_name="valor",
)
sns.barplot(data=plot_df, x="faixa_iaa", y="valor", hue="indicador")
plt.title("Coerencia entre IAA e desempenho/engajamento")
plt.ylabel("Média")
plt.xlabel("Faixa de IAA")
plt.show()

display(Markdown(
    "**Leitura dos dados / interpretação executiva:** a comparação por quartis de IAA mostra se a auto-percepção "
    "está alinhada (ou não) com desempenho e engajamento observados."
))

display(Markdown(
    "**Implicação prática:** quando houver desalinhamento relevante, combinar ação pedagógica "
    "com acompanhamento socioemocional."
))


## Pergunta 5 — Sinais psicossociais e risco futuro (IPS)

**Pergunta de negócio:**  
Existem padrões de IPS associados a maior risco ou queda futura de desempenho?

**Evidência analisada:**  
Quartis de `ips`, `target_risco` e `delta_ida_next`.

**Tabela/gráfico:**  
Tabela de risco médio e delta futuro de IDA por faixa de IPS, com gráficos comparativos.

**Leitura dos dados:**  
A leitura verifica se diferentes faixas psicossociais se associam a maior risco médio ou deterioração futura de desempenho.

**Implicação prática:**  
O IPS deve ser tratado como sinal complementar na triagem preventiva, combinado com leitura pedagógica e psicossocial da equipe.

In [ ]:
q5 = gold_eda[["ips", "delta_ida_next", "target_risco"]].dropna(subset=["ips"]).copy()
q5["faixa_ips"] = safe_qcut(q5["ips"], label="IPS", q=4)

agg_q5 = q5.groupby("faixa_ips", as_index=False).agg(
    risco_medio=("target_risco", "mean"),
    delta_ida_medio=("delta_ida_next", "mean"),
    n=("target_risco", "size"),
)
agg_q5["risco_medio"] = agg_q5["risco_medio"] * 100

display(agg_q5.round(3))

fig, ax1 = plt.subplots(figsize=(10, 4))
sns.barplot(data=agg_q5, x="faixa_ips", y="risco_medio", color="#2a9d8f", ax=ax1)
ax1.set_ylabel("Risco médio (%)")
ax1.set_xlabel("Faixa de IPS")
ax1.set_title("IPS vs risco e queda futura de IDA")

ax2 = ax1.twinx()
sns.lineplot(data=agg_q5, x="faixa_ips", y="delta_ida_medio", marker="o", color="#e76f51", ax=ax2)
ax2.set_ylabel("Delta IDA futuro médio")
plt.show()

display(Markdown(
    "**Leitura dos dados / interpretação executiva:** o cruzamento IPS x risco evidência se variações psicossociais "
    "antecedem deterioração acadêmica."
))

display(Markdown(
    "**Implicação prática:** usar IPS como sinal de alerta complementar para triagem preventiva."
))


## Pergunta 6 — Psicopedagógico e defasagem (IPP e IAN)

**Pergunta de negócio:**  
O IPP confirma, complementa ou contradiz a defasagem indicada pelo IAN?

**Evidência analisada:**  
Distribuição de `ipp` por faixas de `ian`.

**Tabela/gráfico:**  
Tabela e boxplot de IPP por faixa de adequação/defasagem.

**Leitura dos dados:**  
A comparação mostra se os sinais psicopedagógicos convergem com as faixas de defasagem.

**Implicação prática:**  
Casos em que IAN e IPP apontam risco simultaneamente podem receber prioridade no plano de acompanhamento.

In [ ]:
q6 = add_ian_band(silver_eda[["ian", "ipp"]].dropna())

tab_q6 = q6.groupby("faixa_ian", as_index=False).agg(
    ian_medio=("ian", "mean"),
    ipp_medio=("ipp", "mean"),
    n=("ipp", "size"),
)
display(tab_q6.round(3))

plt.figure(figsize=(9, 4))
sns.boxplot(data=q6, x="faixa_ian", y="ipp")
plt.title("Distribuição de IPP por faixa de IAN")
plt.xlabel("Faixa IAN")
plt.ylabel("IPP")
plt.show()

display(Markdown(
    "**Leitura dos dados / interpretação executiva:** comparar IPP entre faixas de IAN ajuda a avaliar convergência "
    "entre sinais de defasagem acadêmica e leitura psicopedagógica."
))

display(Markdown(
    "**Implicação prática:** priorizar casos em que IAN e IPP apontam risco simultaneamente."
))


## Pergunta 7 — Ponto de virada e fatores associados (IPV)

**Pergunta de negócio:**  
Quais indicadores ajudam a explicar variações no ponto de virada do aluno?

**Evidência analisada:**  
Indicadores `ida`, `ieg`, `iaa`, `ips`, `ipp`, `ian`, `defasagem` e `inde` usados em modelo explicativo de IPV.

**Tabela/gráfico:**  
Importância de variáveis do modelo explicativo e evolução anual do IPV médio.

**Leitura dos dados:**  
A leitura identifica quais variáveis têm maior peso explicativo para IPV dentro do recorte disponível.

**Implicação prática:**  
Esses fatores ajudam a orientar quais dimensões acompanhar quando o objetivo é estimular mudança positiva na trajetória do aluno.

In [ ]:
features_q7 = ["ida", "ieg", "iaa", "ips", "ipp", "ian", "defasagem", "inde"]
q7 = silver_eda[features_q7 + ["ipv", "ano_referencia"]].dropna(subset=["ipv"]).copy()

X_q7 = q7[features_q7]
y_q7 = q7["ipv"]

X_q7_imp = SimpleImputer(strategy="median").fit_transform(X_q7)
rf_reg = RandomForestRegressor(n_estimators=300, random_state=42)
rf_reg.fit(X_q7_imp, y_q7)

imp_q7 = pd.DataFrame(
    {"feature": features_q7, "importancia": rf_reg.feature_importances_}
).sort_values("importancia", ascending=False)
display(imp_q7)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.barplot(data=imp_q7.head(8), x="importancia", y="feature", ax=axes[0])
axes[0].set_title("Importancia para explicar IPV")

pv_ano = q7.groupby("ano_referencia", as_index=False)["ipv"].mean()
sns.lineplot(data=pv_ano, x="ano_referencia", y="ipv", marker="o", ax=axes[1])
axes[1].set_title("Evolução do IPV médio")
axes[1].set_ylabel("IPV médio")
plt.tight_layout()
plt.show()

top_feat = imp_q7.head(3)["feature"].tolist()
story_facts["q7_top_features"] = top_feat

display(Markdown(
    f"**Leitura dos dados / interpretação executiva:** os fatores com maior peso explicativo para IPV foram {top_feat}."
))

display(Markdown(
    "**Implicação prática:** orientar monitoramento ativo desses fatores no acompanhamento de rotina."
))


## Pergunta 8 — Multidimensionalidade dos indicadores e INDE

**Pergunta de negócio:**  
Quais combinações de IDA, IEG, IPS e IPP estão associadas a maior INDE?

**Evidência analisada:**  
Classificação alto/baixo por mediana dos indicadores `ida`, `ieg`, `ips` e `ipp`, comparada ao `inde` médio.

**Tabela/gráfico:**  
Tabela de perfis multidimensionais e gráfico das combinações com maior INDE médio.

**Leitura dos dados:**  
A leitura mostra se combinações de indicadores explicam melhor o desenvolvimento do que uma métrica isolada.

**Implicação prática:**  
A gestão deve priorizar perfis compostos, não apenas indicadores individuais.

In [ ]:
q8 = silver_eda[["ida", "ieg", "ips", "ipp", "inde"]].dropna().copy()
for c in ["ida", "ieg", "ips", "ipp"]:
    q8[f"{c}_nivel"] = np.where(q8[c] >= q8[c].median(), "Alto", "Baixo")

q8["combinacao"] = (
    "IDA_" + q8["ida_nivel"]
    + " | IEG_" + q8["ieg_nivel"]
    + " | IPS_" + q8["ips_nivel"]
    + " | IPP_" + q8["ipp_nivel"]
)

agg_q8 = (
    q8.groupby("combinacao", as_index=False)
    .agg(inde_medio=("inde", "mean"), n=("inde", "size"))
    .query("n >= 30")
    .sort_values("inde_medio", ascending=False)
)

display(agg_q8.head(10).round(3))

plt.figure(figsize=(12, 5))
sns.barplot(data=agg_q8.head(10), y="combinacao", x="inde_medio", palette="Blues_r")
plt.title("Top combinações com maior INDE médio (n >= 30)")
plt.xlabel("INDE médio")
plt.ylabel("Combinação")
plt.show()

display(Markdown(
    "**Leitura dos dados / interpretação executiva:** o INDE responde melhor a combinações de indicadores "
    "do que a leitura isolada de uma única métrica."
))

display(Markdown(
    "**Implicação prática:** priorizar gestão por perfis multidimensionais, não por indicador único."
))


## Pergunta 9 — Sinais exploratórios de risco futuro

**Pergunta de negócio:**  
Quais sinais diferenciam alunos com risco futuro dos demais?

**Evidência analisada:**  
Diferença de médias dos indicadores entre `target_risco = 0` e `target_risco = 1`.

**Tabela/gráfico:**  
Tabela comparativa entre classes de risco e gráfico das maiores diferenças de média.

**Leitura dos dados:**  
A leitura identifica sinais associados ao risco, mas ainda em perspectiva exploratória, sem substituir a modelagem preditiva.

**Implicação prática:**  
Esses sinais orientam quais variáveis devem entrar na modelagem e no painel de priorização preventiva.

In [ ]:
cols_q9 = [
    "target_risco", "ian", "ida", "ieg", "iaa", "ips", "ipp", "ipv", "inde", "defasagem",
    "delta_ida_next", "delta_ian_next"
]
q9 = gold_eda[cols_q9].copy()

stats_q9 = q9.groupby("target_risco").mean(numeric_only=True).T
stats_q9.columns = ["sem_risco_0", "com_risco_1"]
stats_q9["dif_1_menos_0"] = stats_q9["com_risco_1"] - stats_q9["sem_risco_0"]
stats_q9 = stats_q9.sort_values("dif_1_menos_0", ascending=False)

display(stats_q9.round(3))

plt.figure(figsize=(10, 6))
plot_q9 = stats_q9[["dif_1_menos_0"]].sort_values("dif_1_menos_0")
sns.barplot(data=plot_q9.reset_index(), x="dif_1_menos_0", y="index", color="#264653")
plt.title("Sinais exploratorios associados ao risco (classe 1 - classe 0)")
plt.xlabel("Diferença de média")
plt.ylabel("Indicador")
plt.show()

top_up = stats_q9.head(3).index.tolist()
top_down = stats_q9.tail(3).index.tolist()
story_facts["q9_top_up"] = top_up
story_facts["q9_top_down"] = top_down

display(Markdown(
    f"**Leitura dos dados / interpretação executiva:** os sinais mais altos na classe de risco foram {top_up}; "
    f"os mais baixos foram {top_down}."
))

display(Markdown(
    "**Implicação prática:** esses sinais orientam a definição de variáveis de entrada para o modelo preditivo."
))


## Pergunta 10 — Efetividade por pedra e trajetória de desenvolvimento

**Pergunta de negócio:**  
Há sinais consistentes de melhora nas pedras/fases do programa?

**Evidência analisada:**  
Evolução de `inde` médio e `target_risco` por `pedra` e `ano_referencia`.

**Tabela/gráfico:**  
Tabela por pedra e ano, com linhas de INDE médio e risco médio.

**Leitura dos dados:**  
A leitura considera diferenças entre grupos e anos, sem forçar tendência quando a cobertura ou comparabilidade forem limitadas.

**Implicação prática:**  
A Passos Mágicos pode usar esse recorte para ajustar estratégias por grupo de pedra e monitorar efeitos ao longo do tempo.

In [ ]:
q10 = gold_eda[["ano_referencia", "pedra", "inde", "target_risco"]].dropna(subset=["pedra"]).copy()

agg_q10 = q10.groupby(["ano_referencia", "pedra"], as_index=False).agg(
    inde_medio=("inde", "mean"),
    risco_medio=("target_risco", "mean"),
    n=("target_risco", "size"),
)
agg_q10["risco_medio"] = agg_q10["risco_medio"] * 100

display(agg_q10.sort_values(["ano_referencia", "pedra"]).round(3))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.lineplot(data=agg_q10, x="ano_referencia", y="inde_medio", hue="pedra", marker="o", ax=axes[0])
axes[0].set_title("INDE médio por pedra ao longo do tempo")

sns.lineplot(data=agg_q10, x="ano_referencia", y="risco_medio", hue="pedra", marker="o", ax=axes[1])
axes[1].set_title("Risco médio (%) por pedra ao longo do tempo")
axes[1].set_ylabel("Risco médio (%)")
plt.tight_layout()
plt.show()

display(Markdown(
    "**Leitura dos dados / interpretação executiva:** a leitura por pedra mostra onde os ganhos são mais consistentes "
    "e onde o risco permanece mais elevado ao longo do ciclo."
))

display(Markdown(
    "**Implicação prática:** ajustar estratégia de acompanhamento por grupo de pedra, "
    "com metas especificas para reducao de risco."
))


## Pergunta 11 — Segmentação de risco e ações práticas

**Pergunta de negócio:**  
Que segmentos de alunos concentram maior risco e quais ações práticas podem ampliar o impacto da análise?

**Evidência analisada:**  
Segmentação por combinações de quartis de `ian`, `ida` e `ieg`, com risco médio e INDE médio.

**Tabela/gráfico:**  
Ranking de segmentos com maior risco médio e tabela de recomendações acionáveis.

**Leitura dos dados:**  
A leitura busca grupos com volume mínimo suficiente para apoiar priorização, sem transformar segmento em regra automática.

**Implicação prática:**  
A segmentação ajuda a transformar a EDA em rotina de acompanhamento, priorização e intervenção preventiva.

In [ ]:
q11 = gold_eda[["ano_referencia", "ian", "ida", "ieg", "ips", "ipp", "inde", "target_risco"]].copy()

for c in ["ian", "ida", "ieg", "ips", "ipp", "inde"]:
    valid = q11[c].dropna()
    if valid.nunique() <= 1:
        q11[f"{c}_q"] = "Q1"
    else:
        q_eff = min(4, int(valid.nunique()))
        labels = [f"Q{i}" for i in range(1, q_eff + 1)]
        ranked = q11[c].rank(method="first")
        q11[f"{c}_q"] = pd.qcut(ranked, q=q_eff, labels=labels)

q11["segmento"] = (
    "IAN_" + q11["ian_q"].astype(str)
    + " | IDA_" + q11["ida_q"].astype(str)
    + " | IEG_" + q11["ieg_q"].astype(str)
)

seg = q11.groupby("segmento", as_index=False).agg(
    risco_medio=("target_risco", "mean"),
    inde_medio=("inde", "mean"),
    n=("target_risco", "size"),
)
seg = seg.query("n >= 40").sort_values("risco_medio", ascending=False)
seg["risco_medio"] = seg["risco_medio"] * 100

display(seg.head(10).round(3))

plt.figure(figsize=(12, 5))
sns.barplot(data=seg.head(10), y="segmento", x="risco_medio", palette="Reds_r")
plt.title("Top segmentos com maior risco médio (n >= 40)")
plt.xlabel("Risco médio (%)")
plt.ylabel("Segmento")
plt.show()

sugestões = pd.DataFrame(
    {
        "frente": [
            "Monitoramento preventivo",
            "Ação acadêmica focada",
            "Apoio psicossocial/psicopedagógico",
            "Rotina de engajamento",
        ],
        "acao_pratica": [
            "Listar semanalmente top 15% por score de risco e abrir plano individual.",
            "Trilhas de reforço para grupos com queda de IDA e baixa adequação.",
            "Priorizar acompanhamento IPS/IPP em segmentos com risco crescente.",
            "Acompanhar IEG quinzenal com gatilho abaixo do percentil 25.",
        ],
    }
)

display(sugestões)

display(Markdown(
    "**Leitura dos dados / interpretação executiva:** os segmentos de maior risco permitem ação focalizada e acompanhamento mais eficiente."
))

display(Markdown(
    "**Implicação prática:** usar segmentação para priorizar atendimento, com revisão periódica de impacto."
))


## Síntese executiva dos achados

A EDA mostra que a Passos Mágicos possui uma base rica para acompanhar a trajetória dos alunos em múltiplas dimensões: desempenho acadêmico, engajamento, aspectos psicossociais, psicopedagógicos e indicadores compostos.

Os principais achados são:

1. A base permite acompanhar alunos por ano, indicadores e grupos, mas exige cuidado metodológico antes de comparar categorias ao longo do tempo.
2. A padronização da variável de fase foi essencial: a coluna original `fase` misturava formatos numéricos, descrições textuais e turmas, especialmente em 2024.
3. A criação de `fase_analitica` reduziu a fragmentação e tornou a comparação temporal do IDA mais confiável, sem preencher ausências artificialmente.
4. IDA, IEG, IAN, IPS, IPP, IPV e INDE mostram diferenças relevantes entre grupos e ajudam a enxergar risco educacional como fenômeno multifatorial.
5. Nem toda variação deve ser interpretada como tendência: mudanças de nomenclatura, cobertura por ano e disponibilidade de indicadores precisam ser consideradas.
6. A EDA justifica a evolução para modelagem de risco, com foco em priorização preventiva e apoio à decisão.

In [ ]:
display(Markdown(
    f"**Resumo de negócio:** taxa média de risco observada na base Gold = {story_facts['taxa_risco_media']:.1f}% "
    f"em {story_facts['n_alunos']} alunos únicos."
))

display(Markdown(
    f"**Sinais mais associados ao risco (classe 1 vs 0):** {story_facts.get('q9_top_up', [])}."
))

display(Markdown(
    f"**Fatores mais relevantes para IPV (modelo explicativo):** {story_facts.get('q7_top_features', [])}."
))


## Recomendações de negócio

1. Monitorar periodicamente os indicadores por aluno, ano e `fase_analitica`.
2. Usar evolução temporal como sinal de atenção, não apenas o valor isolado de um indicador.
3. Priorizar alunos com queda de desempenho, queda de engajamento ou deterioração em indicadores compostos.
4. Tratar o score de risco como ferramenta de triagem e apoio à decisão, não como decisão automática.
5. Combinar leitura quantitativa com avaliação pedagógica, psicossocial e psicopedagógica da equipe.
6. Revisar periodicamente a qualidade das variáveis categóricas, especialmente quando houver mudança de nomenclatura entre anos.
7. Acompanhar o resultado das intervenções para retroalimentar a base e melhorar o processo de priorização.

## Ponte para a modelagem de risco

A análise exploratória mostrou que os dados da Passos Mágicos permitem construir uma visão integrada da trajetória dos alunos, mas também evidenciou a necessidade de padronização e cuidado metodológico. A partir dessa base analítica, o próximo passo é transformar os sinais observados em um modelo de risco, capaz de apoiar a priorização de alunos que precisam de acompanhamento preventivo.

O notebook `03_modelagem_risco.ipynb` parte desta EDA para estruturar variáveis explicativas, treinar modelos e gerar um score de risco. Esse score deve ser entendido como instrumento de apoio à triagem, sempre combinado com a avaliação pedagógica e psicossocial da equipe.

> O principal valor do projeto não está apenas em prever risco, mas em transformar dados educacionais dispersos em uma ferramenta de cuidado preventivo. O modelo ajuda a Passos Mágicos a olhar antes, priorizar melhor e agir com mais evidência.